# CFP_Moirai_Forecasts

Generates 1-step-ahead VaR and ES forecasts using Salesforce Moirai 2.0 Small (11.4M) via mixture distribution sampling. 1000 samples per day; extracts VaR quantiles, empirical ES at 2.5% (FRTB), and fitted Student-t parameters. Seed: 42.

**Paper:** Pele, Lessmann, Hardle (2026)

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # GPU 1, paralel cu Lag-Llama pe GPU 0
print(f"CUDA_VISIBLE_DEVICES set to: {os.environ['CUDA_VISIBLE_DEVICES']}")

CUDA_VISIBLE_DEVICES set to: 1


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path('cfp_ijf_data')
RET_DIR = DATA_DIR / 'returns'

ASSETS = [
    'SP500','STOXX','GDAXI','FCHI','FTSE100','ICLN','NIKKEI','HSI','BOVESPA','NIFTY','ASX200',
    'CBU0','TLT','IBGL','DJCI','GOLD','WTI','NATGAS','BTC','ETH','EURUSD','GBPUSD','USDJPY','AUDUSD'
]
ALPHAS = [0.01, 0.025, 0.05, 0.10]
CONTEXT = 512
N_SAMPLES = 1000

def load_returns(asset):
    df = pd.read_csv(RET_DIR / f'{asset}.csv', parse_dates=['date']).set_index('date').sort_index()
    df = df[df['log_return'].abs() <= 0.50]
    return df['log_return']

print(f'Assets: {len(ASSETS)}, Context: {CONTEXT}, Samples: {N_SAMPLES}')

Assets: 24, Context: 512, Samples: 1000


In [3]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule
from gluonts.dataset.common import ListDataset

assert torch.cuda.is_available(), "CUDA not available"
print(f"Visible: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Loading Moirai 1.1...")
moirai11_module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small")

forecast_11 = MoiraiForecast(
    module=moirai11_module,
    prediction_length=1,
    context_length=CONTEXT,
    patch_size="auto",
    num_samples=N_SAMPLES,
    target_dim=1,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=0,
)
forecast_11 = forecast_11.to(device).eval()
predictor_11 = forecast_11.create_predictor(batch_size=1)

print(f"Moirai 1.1 on: {next(forecast_11.parameters()).device}")

out_dir_11 = DATA_DIR / "moirai"
out_dir_11.mkdir(parents=True, exist_ok=True)
print(f"Output: {out_dir_11}\n")

label = "Moirai-1.1"
pbar = tqdm(ASSETS, desc=label)
for asset in pbar:
    pbar.set_description(f"{label} → {asset}")
    
    ret = load_returns(asset)
    n = len(ret)
    vals = ret.values
    dates = ret.index
    
    records = []
    
    for t in range(CONTEXT, n):
        ds = ListDataset(
            [{
                "target": vals[t-CONTEXT:t],
                "start": pd.Period(dates[t-CONTEXT], freq="D"),
            }],
            freq="D"
        )
        
        with torch.no_grad():
            forecasts = list(predictor_11.predict(ds))
        
        samples = forecasts[0].samples.flatten()
        
        row = {"date": dates[t]}
        for alpha in ALPHAS:
            row[f"VaR_{alpha}"] = np.percentile(samples, alpha * 100)
        row["mean"] = samples.mean()
        row["std"] = samples.std()
        
        sorted_s = np.sort(samples)
        row["ES_empirical_0.025"] = sorted_s[:25].mean()
        row["ES_empirical_0.01"] = sorted_s[:10].mean()
        row["ES_empirical_0.05"] = sorted_s[:50].mean()
        
        records.append(row)
    
    df_out = pd.DataFrame(records).set_index("date")
    df_out.to_parquet(out_dir_11 / f"{asset}.parquet")

print(f"\n✓ Moirai-1.1: {len(ASSETS)} assets saved to {out_dir_11}")

del forecast_11, moirai11_module, predictor_11
torch.cuda.empty_cache()

/home/jovyan/.conda-envs/cfp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Visible: NVIDIA A30
Loading Moirai 1.1...
Moirai 1.1 on: cuda:0
Output: cfp_ijf_data/moirai



Moirai-1.1 → AUDUSD: 100%|██████████| 24/24 [5:45:24<00:00, 863.54s/it]   


✓ Moirai-1.1: 24 assets saved to cfp_ijf_data/moirai


In [4]:
from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module
from scipy.optimize import minimize
from scipy.stats import t as student_t


def fit_student_t_to_quantile_grid(probs, quantiles):
    probs = np.asarray(probs, dtype=float)
    quantiles = np.asarray(quantiles, dtype=float)
    
    mask = np.isfinite(probs) & np.isfinite(quantiles)
    probs = probs[mask]
    quantiles = quantiles[mask]
    
    if len(probs) < 4:
        return np.nan, np.nan, np.nan, False
    
    loc0 = np.median(quantiles)
    scale0 = max(np.std(quantiles), 1e-6)
    df0 = 8.0
    
    def unpack(theta):
        loc = theta[0]
        scale = np.exp(theta[1])
        df = 2.01 + np.exp(theta[2])
        return df, loc, scale
    
    def objective(theta):
        df, loc, scale = unpack(theta)
        try:
            fitted_q = student_t.ppf(probs, df=df, loc=loc, scale=scale)
            return np.mean((quantiles - fitted_q) ** 2)
        except:
            return 1e10
    
    theta0 = np.array([loc0, np.log(scale0), np.log(df0 - 2.01)])
    res = minimize(objective, theta0, method="Nelder-Mead",
                   options={"maxiter": 1000, "xatol": 1e-6})
    
    df, loc, scale = unpack(res.x)
    return df, loc, scale, res.success


print("Loading Moirai 2.0...")
moirai20_module = Moirai2Module.from_pretrained("Salesforce/moirai-2.0-R-small")

forecast_20 = Moirai2Forecast(
    module=moirai20_module,
    prediction_length=1,
    context_length=CONTEXT,
    target_dim=1,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=0,
)
forecast_20 = forecast_20.to(device).eval()
predictor_20 = forecast_20.create_predictor(batch_size=1, device="cuda")

print(f"Moirai 2.0 on: {next(forecast_20.parameters()).device}")

# Sanity check: quantile keys
ret_test = load_returns(ASSETS[0])
ds_test = ListDataset(
    [{"target": ret_test.values[:CONTEXT],
      "start": pd.Period(ret_test.index[0], freq="D")}],
    freq="D"
)
with torch.no_grad():
    fc_test = list(predictor_20.predict(ds_test))[0]

QUANTILE_PROBS = np.array([float(k) for k in fc_test.forecast_keys])
print(f"Moirai 2.0 quantile probs: {QUANTILE_PROBS}")

out_dir_20 = DATA_DIR / "moirai2"
out_dir_20.mkdir(parents=True, exist_ok=True)
print(f"Output: {out_dir_20}\n")

label = "Moirai-2.0"
pbar = tqdm(ASSETS, desc=label)
for asset in pbar:
    pbar.set_description(f"{label} → {asset}")
    
    ret = load_returns(asset)
    n = len(ret)
    vals = ret.values
    dates = ret.index
    
    records = []
    
    for t in range(CONTEXT, n):
        ds = ListDataset(
            [{
                "target": vals[t-CONTEXT:t],
                "start": pd.Period(dates[t-CONTEXT], freq="D"),
            }],
            freq="D"
        )
        
        with torch.no_grad():
            fc = list(predictor_20.predict(ds))[0]
        
        quantiles_pred = fc.forecast_array.flatten()
        
        df_fit, mu_fit, sigma_fit, fit_success = fit_student_t_to_quantile_grid(
            QUANTILE_PROBS, quantiles_pred
        )
        if not fit_success or df_fit < 2:
            df_fit = 2.01
        
        row = {"date": dates[t]}
        for alpha in ALPHAS:
            row[f"VaR_{alpha}"] = -student_t.ppf(
                alpha, df_fit, loc=mu_fit, scale=sigma_fit
            )
        
        row["mean"] = mu_fit
        row["std"] = sigma_fit
        row["df_student"] = df_fit
        
        t_alpha = student_t.ppf(0.025, df_fit)
        tau_alpha = student_t.pdf(t_alpha, df_fit)
        row["ES_student_0.025"] = -(
            mu_fit - sigma_fit * tau_alpha
            * (df_fit + t_alpha**2)
            / ((df_fit - 1) * 0.025)
        )
        
        records.append(row)
    
    df_out = pd.DataFrame(records).set_index("date")
    df_out.to_parquet(out_dir_20 / f"{asset}.parquet")

print(f"\n✓ Moirai-2.0: {len(ASSETS)} assets saved to {out_dir_20}")

del forecast_20, moirai20_module, predictor_20
torch.cuda.empty_cache()

Loading Moirai 2.0...
Moirai 2.0 on: cuda:0
Moirai 2.0 quantile probs: [0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]
Output: cfp_ijf_data/moirai2



Moirai-2.0 → SP500:   0%|          | 0/24 [00:00<?, ?it/s]/tmp/ipykernel_3271/2840202473.py:24: RuntimeWarning: overflow encountered in exp
  df = 2.01 + np.exp(theta[2])
/tmp/ipykernel_3271/2840202473.py:121: RuntimeWarning: invalid value encountered in scalar divide
  mu_fit - sigma_fit * tau_alpha
Moirai-2.0 → GDAXI:   8%|▊         | 2/24 [07:58<1:25:56, 234.41s/it]/tmp/ipykernel_3271/2840202473.py:24: RuntimeWarning: overflow encountered in exp
  df = 2.01 + np.exp(theta[2])
/tmp/ipykernel_3271/2840202473.py:121: RuntimeWarning: invalid value encountered in scalar divide
  mu_fit - sigma_fit * tau_alpha
Moirai-2.0 → FCHI:  12%|█▎        | 3/24 [12:31<1:28:08, 251.82s/it] /tmp/ipykernel_3271/2840202473.py:24: RuntimeWarning: overflow encountered in exp
  df = 2.01 + np.exp(theta[2])
/tmp/ipykernel_3271/2840202473.py:121: RuntimeWarning: invalid value encountered in scalar divide
  mu_fit - sigma_fit * tau_alpha
Moirai-2.0 → FTSE100:  17%|█▋        | 4/24 [17:13<1:28:01, 264.05s/it]/


✓ Moirai-2.0: 24 assets saved to cfp_ijf_data/moirai2
